### Goal: Create Evals For A RAG vs. Graph RAG Agent

What I think I need v1:
1. Create a dataset (COMPLETE)
2. Take dataset and extract data
3. NER for game-specific entities
4. Perform relation extraction to find conextion between entities
5. Design graph schema and relationship types
6. Setup Neo4j
7. Create and configure Neo4j
8. Create embeddings
9. Implement community detection
10. Setup RAG
11. Create evaluation test set

Other things I need:
1. Add key links that are not in the sitemap (ex. abilities and I think trinkets aren't there for some reason)
2. No NER and relation extraction in the sense I was thinking of. Instead create a class/classes for the schema. I have to create the ontology myself
3. Use a class to create the ontology
4. In the class, also make it have the source file so when we get to the RAG step, we can just use the raw HTML for comparison
5. Once we get this complete, then it's learning step 6 & beyond

# Module Imports

In [ ]:
import asyncio
from asyncio import Semaphore
from bs4 import BeautifulSoup, Comment
import aiohttp
import requests

In [ ]:
from utils.scraper import scrape_sitemap
from utils.utils import generate_directories

In [ ]:
# Generate directories needed for project
# TODO: Get rid of this, generate the directories in the function itself
generate_directories()

### Scrape Sitemap & Save Locally
This will be used to embed the raw HTMLs

#### TODO: 
1. Make an orchestration function for this 
2. Save sitemap locally so I don't have to keep rerunning and scraping stuff I already have (for now)
3. Add missing links that I know we somehow missed in the site map

In [ ]:
from utils.scraper import process_sitemap_urls, scrape_multiple_links, save_raw_html_outputs

In [ ]:
sitemap_url = 'https://wildfrostwiki.com/sitemap.xml'
sitemap_urls = scrape_sitemap(sitemap_url)

In [ ]:
urls = process_sitemap_urls(sitemap_urls)

In [ ]:
html_outputs = await scrape_multiple_links(urls)

In [ ]:
raw_html_subdirectory = 'raw_htmls'
save_raw_html_outputs(html_outputs, raw_html_subdirectory)

### Scrape and save the sites in accordance to the ontology

Schemas:
1. Cards Schema: https://wildfrostwiki.com/index.php?title=Baby_Snowbo; grab the table at the end, create a folder structure based on that
2. Fights & Boss Battles: https://wildfrostwiki.com/The_Bog_Berries; grab the Map Events Table at the end
3. Charms: https://wildfrostwiki.com/Charms; another table
4. Stats, Buffs, Debuffs: https://wildfrostwiki.com/Stats
5. Keywords: https://wildfrostwiki.com/Keywords; relations between stats

In [ ]:
import os
import re
from dataclasses import dataclass
from enum import Enum
from typing import Optional
from pathlib import Path
from bs4 import BeautifulSoup, Comment
import logging

logger = logging.getLogger(__name__)

class CardType(Enum):
    """
    What type is the card
    """
    LEADER = 'leaders'
    PETS = 'pets'
    COMPANIONS = 'companions'
    SHADES = 'shades'
    CLUNKERS = 'clunkers'
    ITEMS = 'items'
    ENEMIES = 'enemies'
    ENEMY_CLUNKERS = 'enemy_clunkers'
    MINIBOSSES = 'minibosses'
    BOSSES = 'bosses'

@dataclass
class CardInfo:
    card_name: str
    card_type: CardType
    card_url: str
    card_html: Optional[str] = None

    # Card Stats
    card_description: Optional[str] = None
    health: Optional[int] = None
    attack: Optional[int] = None
    scrap: Optional[int] = None  # Alternative to Heatlh
    counter: Optional[int] = None
    other_stats: Optional[str] = None
  
    
    def sanitized_name(self) -> str:
        """Get sanitized card name safe for filenames"""
        return re.sub(r'[\\/:*?"<>|]', '', self.card_name)


    def save_path(self) -> str:
        """Generate the save path for this card's HTML"""
        return  f'data/structured_outputs/{self.card_type.value}/{self.sanitized_name()}.html'


    def save_html(self) -> bool:
        """
        Save the card's HTML to file with proper directory creation and cleaning
        
        Returns:
            bool: True is saved sucessfully, False otherwise
        """

        if self.card_html is None:
            logger.warning(f"No HTML content to save for {self.card_name}")
            return False
        
        try:
            # Create directory if it doesn't exist
            save_path = Path(self.save_path())
            save_path.parent.mkdir(parents=True, exist_ok=True)

            # Get HTML file
            soup = BeautifulSoup(self.card_html, 'html.parser')

            # Remove comments in HTML
            comments = soup.find_all(string=lambda text: isinstance(text, Comment))
            for comment in comments:
                comment.extract()

            with open(save_path, 'w', encoding='utf-8') as f:
                f.write(soup.prettify())

        except Exception as e:
            logger.error(f'Failed to save HTML for {self.card_name}: {e}')


    def parse_html(self) -> bool:
        """
        Parse the HTML and populate the card stats fields
        
        Returns:
            bool: True if parsing succeeded, False otherwise
        """
        if not self.card_html:
            logger.warning(f"No HTML content to parse for {self.card_name}")
            return False
        
        try:
            soup = BeautifulSoup(self.card_html, 'html.parser')
            
            # Extract description
            description_tag = soup.find("meta", attrs={'name': 'description'})
            if description_tag:
                self.card_description = description_tag.get("content", "")
            
            # Extract stats from infobox
            infobox = soup.find('table', {'id': 'infobox'})
            if infobox:
                rows = infobox.find_all('tr')
                if len(rows) >= 4:
                    # Stats are in rows 2 and 3
                    stats_headers = [th.text.strip() for th in rows[2].find_all('th')]
                    stats_values = [td.text.strip() for td in rows[3].find_all('td')]
                    
                    stats = dict(zip(stats_headers, stats_values))
                    
                    # Dynamically populate matching fields
                    for stat_name, value in stats.items():
                        attr_name = stat_name.lower()
                        
                        if hasattr(self, attr_name):
                            if value.strip() == "":
                                setattr(self, attr_name, None)
                            elif value.isdigit():
                                setattr(self, attr_name, int(value))
                            else:
                                setattr(self, attr_name, value)
                    
                    # Look for "Other Stats" section (effects)
                    for i, row in enumerate(rows):
                        th = row.find('th')
                        if th and th.text.strip() == "Other Stats":
                            if i + 1 < len(rows):
                                other_stats_row = rows[i + 1]
                                td = other_stats_row.find('td')
                                if td:
                                    other_stats_text = td.get_text(strip=True)
                                    self.other_stats = other_stats_text if other_stats_text else None
                            break
            
            return True
            
        except Exception as e:
            logger.error(f"Failed to parse HTML for {self.card_name}: {e}")
            return False
    
    def __str__(self) -> str:
        """String representation of the card"""
        lines = [
            f"Card Name: {self.card_name}",
            f"Card Description: {self.card_description}",
            f"Card Type: {self.card_type.value}",
            f"Health: {self.health}",
            f"Attack: {self.attack}",
            f"Scrap: {self.scrap}",
            f"Counter: {self.counter}",
            f"Other Stats: {self.other_stats}",
            f"Generic Ability: **TODO**",
            f"Specific Ability: **TODO**"
        ]
        return "\n".join(lines)


In [ ]:
import requests
from bs4 import BeautifulSoup

def generate_card_type_html_schema(schema_url: str='https://wildfrostwiki.com/index.php?title=Baby_Snowbo'):
    """
    Given a base schema url, take the cards, break them down into their schema, and save accordingly.

    schema_url (str): A base url to extract the card type schema. Default link is provided
    """
    response = requests.get(schema_url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    card_schema = soup.find('table', {'class': 'wikitable', 'id':'navbox'})

    card_data = {}

    rows = card_schema.find_all('tr')[1:]

    for row in rows:
        card_list = []
        card_type = row.find('th')
        card_type_data = row.find('td')
        card_names = card_type_data.find_all('a')

        category_name = card_type.get_text().strip().lower().replace(' ', '_')

        print(f"Schema Text: {category_name}")

        for n in card_names:
            card_name = n.get_text().strip()
            # print(f'{card_name}')
            card_list.append(card_name)
        
        card_data[category_name] = card_list

    return card_data

In [ ]:
card_type_schema = generate_card_type_html_schema()

In [ ]:
import os
import json

filename = 'data/schemas/'
schema_filename = os.path.join(filename,'card_type_schema.json')
os.makedirs(filename, exist_ok=True)

with open(schema_filename,'w',encoding='utf-8') as f:
    json.dump(card_type_schema, f, indent=4)

In [ ]:
for k, v in card_type_schema.items():
    print(f'{k}: {v}')

In [ ]:
base_url = 'https://wildfrostwiki.com'

In [ ]:
# Need to make sure that the sub_directory is part of the link. Might just use a tuple and use tuple unpacking into the scrape multiple links function
# TODO: 
#   1. Create a dictionary where each card link is placed in a subdirectory as a key 
#   2. Take the dictionary, scrape 

card_infos  = []
for card_type, cards in card_type_schema.items():
    if card_type == 'leaders':
        continue

    for card_name in cards:
        card_info = CardInfo(
            card_name=card_name,
            card_type=CardType(card_type),
            card_url=f'{base_url}/{card_name}'
        )
        card_infos.append(card_info)

for c in card_infos:
    print(f'{c}\n')

In [ ]:
urls = [card.card_url for card in card_infos]

In [ ]:
card_types_html_outputs = await scrape_multiple_links(urls)

In [ ]:
for card_info, html in zip(card_infos, card_types_html_outputs):
    card_info.card_html = html
    if card_info.card_html is not None:
        card_info.save_html()
        card_info.parse_html()

In [ ]:
for c in card_infos:
    print(f'{c}\n')

In [ ]:
binku = card_infos[0]

### Try to get the base stats from each card

In [ ]:
from typing import Dict, Any
from bs4 import BeautifulSoup
import logging

logger = logging.getLogger(__name__)

class PetParser:
    """Parser for pet cards"""
    
    def parse(self, card_info: CardInfo) -> Dict[str, Any]:
        if not card_info.card_html:
            print(f"\nCARD HAS NO HTML: {card_info.card_name}\n")
            return {}
        
        soup = BeautifulSoup(card_info.card_html, 'html.parser')
        
        data = {}
        
        # Extract title
        if soup.title:
            title = soup.title.string
            data['title'] = title.split('-')[0].strip() if title else None
        
        # Extract description
        description_tag = soup.find("meta", attrs={'name': 'description'})
        if description_tag:
            data['description'] = description_tag.get("content", "")
        
        # Extract stats from infobox
        infobox = soup.find('table', {'id': 'infobox'})
        if infobox:
            rows = infobox.find_all('tr')
            if len(rows) >= 4:
                # Stats are in rows 2 and 3
                stats_headers = [th.text.strip() for th in rows[2].find_all('th')]
                stats_values = [td.text.strip() for td in rows[3].find_all('td')]
                
                stats = dict(zip(stats_headers, stats_values))
                
                # Convert to ints where possible
                for stat_name, value in stats.items():
                    if value.isdigit():
                        data[stat_name.lower()] = int(value)
                    elif value.strip() == "":  # Handle empty strings
                        data[stat_name.lower()] = None
                    else:
                        data[stat_name.lower()] = value


                # Look for "Other Stats" section
                for i, row in enumerate(rows):
                    th = row.find('th')
                    if th and th.text.strip() == "Other Stats":
                        # The next row should contain the other stats
                        if i + 1 < len(rows):
                            other_stats_row = rows[i + 1]
                            td = other_stats_row.find('td')
                            if td:
                                other_stats_text = td.get_text(strip=True)
                                data['other_stats'] = other_stats_text if other_stats_text else None
                        break
        
        return data

In [ ]:
pet_parser = PetParser()

In [ ]:
collected_data = []
for card_info in card_infos:
    parsed_data = pet_parser.parse(card_info)
    collected_data.append(parsed_data)

In [ ]:
collected_data

In [ ]:
parsed_data = pet_parser.parse(binku)
print(parsed_data)

In [ ]:
binku.health = parsed_data['health']
binku.attack = parsed_data['attack']
binku.counter = parsed_data['counter']

In [ ]:
binku

### Schema & Ontology Creation
Given the files, let's try to make an schema, then figure out how to bridge that to Neo4j

##### Classes to use a schema, keeping it simple for now

##### Gather the files in data

In [ ]:
import os

folder_path = 'data/raw_htmls'
files = []

for file in os.listdir(folder_path):
    full_path = os.path.join(folder_path, file)
    
    if os.path.isfile(full_path):
        files.append(full_path)

count =0
for f in files:
    print(f'Index: {count}, html: {f}')
    count+=1

print(type(files))

In [ ]:
test_html = files[39]

##### Given the file, let's scrape the card type

In [ ]:
with open(test_html, 'r', encoding='utf-8') as f:
    data = f.read()

In [ ]:
data

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(data, 'html.parser')

In [ ]:
soup

In [ ]:
title = soup.title.string
print(title.split('-')[0].strip())

In [ ]:
description_tag = soup.find("meta", attrs={'name': 'description'})
description = description_tag.get("content")
description

In [ ]:
infobox = soup.find('table', {'id': 'infobox'})
print(infobox)

In [ ]:
rows = infobox.find_all('tr')

In [ ]:
rows

In [ ]:
# Get all rows
rows = infobox.find_all('tr')

# Find the stats - they're in rows 2 and 3
stats_headers = [th.text.strip() for th in rows[2].find_all('th')]
stats_values = [td.text.strip() for td in rows[3].find_all('td')]

# Combine into dict
stats = dict(zip(stats_headers, stats_values))

print(stats)
# Output: {'Health': '1', 'Attack': '1', 'Counter': '2'}

In [ ]:
stat_links_to_find = {
    'Health': None,
    'Attack': None,
    'Counter': None
}

In [ ]:
for stat_name in stat_links_to_find:
    a_tag = infobox.find('a', title=stat_name)
    
    if a_tag:
        stat_links_to_find[stat_name] = a_tag.get('href')

# Now, you have a dictionary with the stat names and their found links
print(stat_links_to_find)
